# 🔍 Notebook 2: Explainability & SHAP for Clinical AI

**Workshop:** Practical Introduction to AI for Mental Health Applications  
**Estimated time:** 20 minutes  
**Prerequisites:** Ideally run Notebook 1 first, but this notebook regenerates everything needed.

## Learning Objectives
1. Understand the difference between global and local explainability
2. Use SHAP TreeExplainer to explain a Random Forest model
3. Interpret beeswarm plots, waterfall plots, and dependence plots
4. Conduct a basic fairness analysis of model explanations across demographic groups

---
> 💡 **Why explainability?** In clinical contexts, a model that cannot explain its decisions cannot be trusted by clinicians, approved by regulators, or audited for bias. SHAP provides mathematically rigorous explanations that work for any ML model.

## 0. Setup & Data Loading

We regenerate the exact same dataset and model from Notebook 1 using the same random seed (42). This ensures results are identical whether you run this notebook standalone or after Notebook 1.

In [ ]:
# Install and import
!pip install shap --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

print("✓ Libraries loaded")

# Regenerate dataset (identical to Notebook 1 with same seed)
def generate_abide_dataset(seed=42):
    np.random.seed(seed)
    N_ASD, N_CTRL = 250, 250

    def correlated_iq(n, mean_fiq=100, std=12):
        corr = np.array([[1.0, 0.7, 0.7], [0.7, 1.0, 0.5], [0.7, 0.5, 1.0]])
        L = np.linalg.cholesky(corr)
        return np.clip(np.random.normal(0, 1, (n, 3)) @ L.T * std + mean_fiq, 70, 145)

    asd_iq = correlated_iq(N_ASD, 100)
    ctrl_iq = correlated_iq(N_CTRL, 110)

    df = pd.DataFrame({
        'AGE_AT_SCAN': np.concatenate([np.random.uniform(6, 58, N_ASD),
                                       np.random.uniform(6, 58, N_CTRL)]),
        'SEX': np.concatenate([np.random.choice([0, 1], N_ASD, p=[0.82, 0.18]),
                                np.random.choice([0, 1], N_CTRL, p=[0.55, 0.45])]),
        'FIQ': np.concatenate([asd_iq[:, 0], ctrl_iq[:, 0]]),
        'VIQ': np.concatenate([asd_iq[:, 1], ctrl_iq[:, 1]]),
        'PIQ': np.concatenate([asd_iq[:, 2], ctrl_iq[:, 2]]),
        'SRS_RAW_TOTAL': np.concatenate([
            np.clip(np.random.normal(120, 30, N_ASD), 0, 195),
            np.clip(np.random.normal(38, 22, N_CTRL), 0, 195)]),
        'ADOS_TOTAL': np.concatenate([
            np.clip(np.random.normal(12, 3, N_ASD), 0, 28).round(),
            np.clip(np.random.normal(2.5, 2, N_CTRL), 0, 15).round()]),
        'SCQ_TOTAL': np.concatenate([
            np.clip(np.random.normal(18, 4, N_ASD), 0, 39).round(),
            np.clip(np.random.normal(5, 3, N_CTRL), 0, 39).round()]),
        'VINELAND_COMMUNICATION_STANDARD_SCORE': np.concatenate([
            np.clip(np.random.normal(82, 15, N_ASD), 40, 160).round(),
            np.clip(np.random.normal(112, 14, N_CTRL), 40, 160).round()]),
        'DX_GROUP': np.concatenate([np.ones(N_ASD), np.zeros(N_CTRL)])
    })
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)

df = generate_abide_dataset()
feature_cols = ['AGE_AT_SCAN', 'SEX', 'FIQ', 'VIQ', 'PIQ',
                'SRS_RAW_TOTAL', 'ADOS_TOTAL', 'SCQ_TOTAL',
                'VINELAND_COMMUNICATION_STANDARD_SCORE']
X = df[feature_cols].copy()
y = df['DX_GROUP'].astype(int)

# Introduce missing values (5%) then preprocess
for col in ['SRS_RAW_TOTAL', 'ADOS_TOTAL', 'SCQ_TOTAL', 'VINELAND_COMMUNICATION_STANDARD_SCORE']:
    X.loc[np.random.random(len(X)) < 0.05, col] = np.nan

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)
X_train_s = pd.DataFrame(scaler.fit_transform(X_train_imp), columns=feature_cols)
X_test_s = pd.DataFrame(scaler.transform(X_test_imp), columns=feature_cols)

# Load or train RF model
if os.path.exists('rf_model.pkl'):
    rf_model = joblib.load('rf_model.pkl')
    print("✓ Loaded trained model from Notebook 1")
else:
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_s, y_train)
    print("✓ Retrained Random Forest model")

y_pred = rf_model.predict(X_test_s)
y_prob = rf_model.predict_proba(X_test_s)[:, 1]
print(f"\nModel accuracy on test set: {(y_pred == y_test.values).mean():.3f}")

## 1. Why Explainability Matters in Mental Health AI

### The Black Box Problem
When a clinician receives a model output like "This child has an 87% probability of ASD," they naturally ask: *Why?* What drove that prediction? Is it the ADOS score, the age, or something unexpected?

Without explainability:
- Clinicians can't verify the model is using clinically valid reasoning
- Biases in the training data remain hidden
- Regulatory approval is nearly impossible (FDA, Health Canada require transparency)
- Clinical accountability is unclear ("Who is responsible when the AI is wrong?")

### Global vs. Local Explanations
| Type | What it answers | When to use |
|------|-----------------|-------------|
| **Global** | "Which features does the model rely on overall?" | Model validation, research reporting |
| **Local** | "Why did the model give THIS patient THIS prediction?" | Clinical decision support, patient communication |

**SHAP (SHapley Additive exPlanations)** provides both. It's grounded in cooperative game theory (Shapley values) and is the current gold standard for ML explainability.

## 2. Global Explainability — Built-in Feature Importance

Random Forest has a built-in feature importance measure: how much does each feature reduce impurity (improve predictions) across all trees?

**Limitation:** This tells us *which* features matter, but not *how* they matter (direction of effect).

In [ ]:
# Built-in Random Forest feature importances
importances = pd.Series(rf_model.feature_importances_, index=feature_cols)
importances_sorted = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#B93680' if imp > importances.median() else '#94A3B8'
          for imp in importances_sorted]
bars = ax.barh(importances_sorted.index, importances_sorted.values,
               color=colors, alpha=0.85)
ax.axvline(importances.median(), color='black', linestyle='--',
           alpha=0.5, label='Median')
ax.set_xlabel('Feature Importance (Mean Decrease in Impurity)', fontsize=12)
ax.set_title('Random Forest Built-in Feature Importances\n(Top features highlighted in pink)',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
for bar, imp in zip(bars, importances_sorted.values):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
            f'{imp:.3f}', va='center', fontsize=10)
plt.tight_layout()
plt.savefig('rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("Top features (built-in importance):")
for feat, imp in importances.sort_values(ascending=False).items():
    direction = "\u2191 ASD" if feat in ['SRS_RAW_TOTAL', 'ADOS_TOTAL', 'SCQ_TOTAL'] else "?"
    print(f"  {feat:<45} {imp:.3f}  ({direction})")
print("\n\u26a0\ufe0f  Limitation: This doesn't tell us the DIRECTION of each feature's effect!")

> 💬 **DISCUSSION POINT:** The feature importance plot shows SRS and ADOS scores are most important. But this doesn't tell us: does a *higher* SRS score push toward ASD or away from ASD? That's what SHAP solves next.

## 3. Introducing SHAP

### The Game Theory Intuition
Imagine a team of doctors making a diagnosis. SHAP asks: "If we removed Doctor A from the team, how would the diagnosis change? What about removing Doctor B?" The average change across all possible team compositions is Doctor A's "Shapley value" — their fair contribution.

SHAP applies this idea to model features:
- Positive SHAP value → feature pushes prediction **toward ASD**
- Negative SHAP value → feature pushes prediction **away from ASD**
- SHAP value of 0 → feature had no effect for this prediction

### Key formula:
`prediction = baseline_value + Σ(SHAP values for all features)`

In [ ]:
%%time
# Create SHAP explainer
# WHY TreeExplainer: optimized for tree-based models (Random Forest, XGBoost)
# Much faster than the generic KernelExplainer
explainer = shap.TreeExplainer(rf_model)

# Compute SHAP values for the test set
# WHY X_test_s (scaled): model was trained on scaled data
shap_values = explainer.shap_values(X_test_s)

# For binary classification, shap_values is a list [class0_values, class1_values]
# We want class 1 (ASD probability)
sv = shap_values[1]  # Shape: (n_test_samples, n_features)

print(f"SHAP values shape: {sv.shape}")
print(f"  Rows: {sv.shape[0]} test patients")
print(f"  Columns: {sv.shape[1]} features")
print(f"\nBaseline (expected) value: {explainer.expected_value[1]:.3f}")
print(f"  (This is the average model prediction across all training samples)")
print(f"\nSample SHAP values for first patient:")
for feat, val in zip(feature_cols, sv[0]):
    direction = "\u2192 ASD" if val > 0 else "\u2192 Control"
    print(f"  {feat:<45} {val:+.3f}  {direction}")

## 4. Global SHAP — The Beeswarm Plot

The beeswarm plot is the standard way to show global SHAP explanations. Each dot is one patient.

**How to read it:**
- **X-axis:** SHAP value (impact on prediction). Right = pushes toward ASD, Left = away from ASD
- **Color:** Feature value (red = high, blue = low)
- **Y-axis:** Features ranked by importance (most important at top)

In [ ]:
# Global SHAP beeswarm plot
plt.figure(figsize=(10, 7))
shap.summary_plot(sv, X_test_s, feature_names=feature_cols,
                   plot_type='dot', max_display=9, show=False)
plt.title('SHAP Beeswarm Plot — Global Feature Importance & Direction',
          fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

print("Reading the beeswarm plot:")
print("  \u2022 SRS_RAW_TOTAL: HIGH values (red) \u2192 positive SHAP \u2192 pushes toward ASD \u2713")
print("  \u2022 VINELAND: HIGH values (red) \u2192 negative SHAP \u2192 pushes AWAY from ASD \u2713")
print("  \u2022 These patterns match clinical expectations \u2014 validation of the model!")

> 🔧 **EXERCISE:** Re-run the SHAP summary plot with `plot_type='bar'` instead of `plot_type='dot'`.
>
> **Hint:** Change `plot_type='dot'` to `plot_type='bar'`. How does this differ from the beeswarm? The bar plot shows mean absolute SHAP value (overall importance) but loses the direction and individual patient information.

## 5. Local Explainability — Explaining Individual Predictions

For clinical decision support, we need to explain *specific* predictions. SHAP waterfall plots show exactly which features pushed a prediction up or down for a single patient.

In [ ]:
# Find three interesting patients in the test set
probs = y_prob  # RF predicted probabilities

# Patient 1: High-confidence ASD prediction
asd_candidates = np.where((y_test.values == 1) & (probs > 0.85))[0]
idx_asd = asd_candidates[0] if len(asd_candidates) > 0 else np.argmax(probs)

# Patient 2: High-confidence Control prediction
ctrl_candidates = np.where((y_test.values == 0) & (probs < 0.15))[0]
idx_ctrl = ctrl_candidates[0] if len(ctrl_candidates) > 0 else np.argmin(probs)

# Patient 3: Borderline/uncertain prediction
uncertain_candidates = np.where(np.abs(probs - 0.5) < 0.1)[0]
idx_uncertain = (uncertain_candidates[0] if len(uncertain_candidates) > 0
                 else np.argmin(np.abs(probs - 0.5)))

print("Selected patients for explanation:")
for label, idx in [("High-confidence ASD", idx_asd),
                    ("High-confidence Control", idx_ctrl),
                    ("Uncertain", idx_uncertain)]:
    true_label = "ASD" if y_test.values[idx] == 1 else "Control"
    print(f"  {label}: Patient #{idx}, True={true_label}, Predicted prob={probs[idx]:.3f}")

In [ ]:
# Waterfall plot for high-confidence ASD patient
fig = plt.figure(figsize=(10, 6))
shap_exp = shap.Explanation(
    values=sv[idx_asd],
    base_values=explainer.expected_value[1],
    data=X_test_s.iloc[idx_asd].values,
    feature_names=feature_cols
)
shap.waterfall_plot(shap_exp, show=False, max_display=9)
true_label_asd = "ASD" if y_test.values[idx_asd] == 1 else "Control"
plt.title(f'Patient {idx_asd}: High-Confidence ASD Prediction (prob={probs[idx_asd]:.3f})\n'
          f'True label: {true_label_asd}',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_waterfall_asd.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Waterfall plot for uncertain patient
fig = plt.figure(figsize=(10, 6))
shap_exp_unc = shap.Explanation(
    values=sv[idx_uncertain],
    base_values=explainer.expected_value[1],
    data=X_test_s.iloc[idx_uncertain].values,
    feature_names=feature_cols
)
shap.waterfall_plot(shap_exp_unc, show=False, max_display=9)
true_label_unc = "ASD" if y_test.values[idx_uncertain] == 1 else "Control"
plt.title(f'Patient {idx_uncertain}: Uncertain Prediction (prob={probs[idx_uncertain]:.3f})\n'
          f'True label: {true_label_unc}',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_waterfall_uncertain.png', dpi=150, bbox_inches='tight')
plt.show()

print("Notice: For uncertain patients, features push in BOTH directions — the model is genuinely unsure.")

> 🔧 **EXERCISE:** Change `idx_asd` or `idx_uncertain` to a different patient index (e.g., try index 5, 20, 50).
>
> **Hint:** Modify the waterfall plot cell above to use a different integer index. Can you find a patient where the model is **wrong**? What features misled it?
>
> 💬 **DISCUSSION POINT:** This is powerful for clinical audit — "Why did the model flag this child?" The waterfall plot gives clinicians a concrete, feature-level justification to evaluate.

## 6. SHAP Dependence Plots — Feature Interactions

SHAP dependence plots show how a feature's SHAP value changes with its actual value — revealing non-linear effects and interactions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: SRS score dependence
ax = axes[0]
srs_vals = X_test_s['SRS_RAW_TOTAL'].values
srs_shap = sv[:, feature_cols.index('SRS_RAW_TOTAL')]
scatter = ax.scatter(srs_vals, srs_shap, c=X_test_s['SEX'].values,
                     cmap='coolwarm', alpha=0.7, s=50)
ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('SRS Raw Total (scaled)', fontsize=12)
ax.set_ylabel('SHAP Value (impact on ASD prediction)', fontsize=12)
ax.set_title('SRS Score \u2192 SHAP Value\n(colored by SEX: blue=Male, red=Female)',
             fontsize=13, fontweight='bold')
plt.colorbar(scatter, ax=ax, label='SEX (0=Male, 1=Female)')

# Plot 2: ADOS score dependence
ax = axes[1]
ados_vals = X_test_s['ADOS_TOTAL'].values
ados_shap = sv[:, feature_cols.index('ADOS_TOTAL')]
scatter2 = ax.scatter(ados_vals, ados_shap, c=X_test_s['AGE_AT_SCAN'].values,
                      cmap='plasma', alpha=0.7, s=50)
ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('ADOS Total (scaled)', fontsize=12)
ax.set_ylabel('SHAP Value (impact on ASD prediction)', fontsize=12)
ax.set_title('ADOS Score \u2192 SHAP Value\n(colored by Age)',
             fontsize=13, fontweight='bold')
plt.colorbar(scatter2, ax=ax, label='Age at Scan (scaled)')

plt.tight_layout()
plt.savefig('shap_dependence_plots.png', dpi=150, bbox_inches='tight')
plt.show()

print("Interpretation:")
print("  \u2022 SRS: monotonically positive \u2014 higher SRS always pushes toward ASD prediction")
print("  \u2022 Color variation: if red dots (female) cluster differently, suggests sex-based interaction")

## 7. Fairness Check — Do Explanations Differ Across Groups?

A fair model should use features in similar ways across demographic groups. If the model relies on very different features for male vs. female patients, this could reflect underlying biases in training data.

This is especially relevant for ASD: the "female autism phenotype" is well-documented — females with ASD often present differently on standardized assessments (less social withdrawal, stronger masking), meaning a model trained mostly on males may not generalize well to females.

In [ ]:
# Fairness analysis: mean |SHAP| by sex
sex_col_idx = feature_cols.index('SEX')
sex_vals = X_test_s['SEX'].values

male_idx = np.where(sex_vals <= 0)[0]   # scaled SEX <= 0 -> Male
female_idx = np.where(sex_vals > 0)[0]  # scaled SEX > 0 -> Female

print(f"Test set: {len(male_idx)} males, {len(female_idx)} females")

# Compute mean absolute SHAP per feature per group
mean_abs_shap_male = np.abs(sv[male_idx]).mean(axis=0)
mean_abs_shap_female = np.abs(sv[female_idx]).mean(axis=0)

# Plot comparison
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(feature_cols))
width = 0.35

bars1 = ax.bar(x - width / 2, mean_abs_shap_male, width, label='Male',
               color='#2563EB', alpha=0.8)
bars2 = ax.bar(x + width / 2, mean_abs_shap_female, width, label='Female',
               color='#B93680', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels([f.replace('_', '\n') for f in feature_cols], fontsize=9)
ax.set_ylabel('Mean |SHAP Value|', fontsize=12)
ax.set_title('Fairness Check: Feature Importance by Sex\n'
             '(Large differences may indicate model bias)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('shap_fairness_bysex.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFairness interpretation:")
print("  If ADOS/SRS importance is much higher for males than females,")
print("  this may reflect that these scales were normed primarily on male ASD presentations.")
print("  The model may be less reliable for female patients \u2014 a critical clinical consideration.")

> 💬 **DISCUSSION POINT:** The "female autism phenotype" is well-documented in clinical research. Females with ASD often:
> - Have stronger social camouflaging ("masking") skills
> - Score lower on ADOS/ADI even when meeting diagnostic criteria
> - Receive diagnosis significantly later on average (age 13 vs age 7 for males)
>
> If our model was trained primarily on males, its SHAP profiles for female patients may look systematically different. What should we do before deploying this model clinically?

## 8. Practical Takeaways & Bridge to Notebook 3

### When to use SHAP in your research:
- **Model validation:** Do the most important features match clinical theory? (Sanity check)
- **Publication:** Report SHAP beeswarm plots alongside AUC/F1 for transparency
- **Clinical reporting:** Use waterfall plots for individual patient explanations
- **Fairness audits:** Compare SHAP profiles across demographic groups before deployment

### Limitations of SHAP:
- ⏱️ Computationally expensive for large datasets and complex models
- 🔢 SHAP values are approximations for some model types (KernelSHAP)
- 🔄 Correlations between features can make individual SHAP values hard to interpret
- 🔬 SHAP explains the model — not necessarily the underlying biology

### What's next?
We have a model, and we can explain its predictions. Now: what if we want to apply LLMs to clinical text — like summarizing clinical notes — but can't send real patient data to a cloud API?

→ **Notebook 3: Privacy-Preserving AI with Local LLMs (Ollama)**